# Extract weather and traffic features for BC sensor data

This notebook reads:
`BC_raw_data.csv`

Expected columns: `datetime`, `lat`, `lon`, `bc`.

It creates:
`Artifacts/extracted_features.csv`

with additional columns:
- `Temperature` (deg C)
- `Humidity` (%)
- `Wind_speed` (km/h)
- TomTom **Flow Segment Data** fields as separate columns prefixed with `TT_` (e.g. `TT_frc`, `TT_currentSpeed`, `TT_freeFlowSpeed`, travel times, `TT_confidence`, `TT_roadClosure`, `TT_coordinates` as JSON, optional `TT_openLr`, plus service metadata keys from the JSON response).
- `Traffic_flow` — same as before: duplicate of `TT_freeFlowSpeed` (km/h) for backward compatibility.

## APIs used
- Open-Meteo Archive API (free, no key): historical weather by location and date.
- TomTom Traffic Flow **Flow Segment Data** endpoint (free tier, API key required): full segment payload at the closest road to each point.


In [ ]:
import json
import os
import time
from datetime import timezone

import numpy as np   # pyright: ignore[reportMissingImports]
import pandas as pd  # pyright: ignore[reportMissingImports]
import requests  # pyright: ignore[reportMissingModuleSource]
from tqdm.auto import tqdm  # pyright: ignore[reportMissingModuleSource]

pd.set_option("display.max_columns", None)
tqdm.pandas()

In [ ]:
from pathlib import Path

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "Code" else cwd

ROOT = _repo_root()
INPUT_CSV = str(ROOT / "Artifacts" / "BC_raw_data.csv")
OUTPUT_DIR = str(ROOT / "Artifacts" / "extracted_feature_chunks")
FINAL_OUTPUT_CSV = str(ROOT / "Artifacts" / "extracted_features.csv")

# Process one chunk at a time (set 1..n, run notebook, then change and rerun)
# TomTom Free Tier has a limit of 2500 requests per day. Chunking helps to spread the requests over time.
N_CHUNKS = 5
CHUNK_TO_RUN = 5

# Set in your shell before Jupyter: export TOMTOM_API_KEY="your_key"
TOMTOM_API_KEY = os.environ.get("TOMTOM_API_KEY", "").strip()

os.makedirs(OUTPUT_DIR, exist_ok=True)
if CHUNK_TO_RUN < 1 or CHUNK_TO_RUN > N_CHUNKS:
    raise ValueError(f"CHUNK_TO_RUN must be between 1 and {N_CHUNKS}")

full_df = pd.read_csv(INPUT_CSV)
expected_cols = {"datetime", "lat", "lon", "bc"}
missing = expected_cols - set(full_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

full_df["datetime"] = pd.to_datetime(full_df["datetime"], utc=True, errors="coerce")
if full_df["datetime"].isna().any():
    bad_rows = full_df[full_df["datetime"].isna()].index[:5].tolist()
    raise ValueError(f"Found invalid datetime rows, first indices: {bad_rows}")

chunks = np.array_split(full_df, N_CHUNKS)
df = chunks[CHUNK_TO_RUN - 1].copy().reset_index(drop=True)
CHUNK_OUTPUT_CSV = os.path.join(OUTPUT_DIR, f"extracted_features_chunk_{CHUNK_TO_RUN:02d}_of_{N_CHUNKS:02d}.csv")

print(f"Total rows in source: {len(full_df)}")
print(f"Running chunk {CHUNK_TO_RUN}/{N_CHUNKS} with {len(df)} rows")
print(f"Chunk output file: {CHUNK_OUTPUT_CSV}")
df.head()

In [ ]:
weather_cache = {}
traffic_cache = {}

weather_session = requests.Session()
traffic_session = requests.Session()

def _safe_get_json(session, url, params, timeout=20):
    resp = session.get(url, params=params, timeout=timeout)
    resp.raise_for_status()
    return resp.json()

def get_weather_open_meteo(lat, lon, ts_utc):
    # Hourly historical data from Open-Meteo archive.
    hour_ts = pd.Timestamp(ts_utc).tz_convert("UTC").floor("h")
    key = (round(float(lat), 4), round(float(lon), 4), hour_ts)
    if key in weather_cache:
        return weather_cache[key]

    date_str = hour_ts.strftime("%Y-%m-%d")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": key[0],
        "longitude": key[1],
        "start_date": date_str,
        "end_date": date_str,
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m",
        "timezone": "UTC",
    }

    try:
        data = _safe_get_json(weather_session, url, params)
        hourly = data.get("hourly", {})
        times = hourly.get("time", [])
        if not times:
            out = (np.nan, np.nan, np.nan)
        else:
            target_str = hour_ts.strftime("%Y-%m-%dT%H:00")
            if target_str in times:
                i = times.index(target_str)
            else:
                i = min(range(len(times)), key=lambda idx: abs(pd.Timestamp(times[idx], tz="UTC") - hour_ts))

            out = (
                hourly.get("temperature_2m", [np.nan] * len(times))[i],
                hourly.get("relative_humidity_2m", [np.nan] * len(times))[i],
                hourly.get("wind_speed_10m", [np.nan] * len(times))[i],
            )
    except Exception:
        out = (np.nan, np.nan, np.nan)

    weather_cache[key] = out
    time.sleep(0.05)
    return out

def _flow_segment_to_columns(flow_data):
    """Flatten TomTom flowSegmentData JSON into one row (TT_* columns)."""
    row = {}
    if not flow_data:
        return row
    for k, v in flow_data.items():
        if k.startswith("-"):
            col = f"TT_{k[1:]}"
        else:
            col = f"TT_{k}"
        if isinstance(v, (dict, list)):
            try:
                row[col] = json.dumps(v, separators=(",", ":"))
            except (TypeError, ValueError):
                row[col] = np.nan
        else:
            row[col] = v
    return row


def get_traffic_tomtom(lat, lon):
    # TomTom Flow Segment Data: all fields returned for the road segment nearest the point.
    loc_key = (round(float(lat), 5), round(float(lon), 5))
    if loc_key in traffic_cache:
        return traffic_cache[loc_key]

    if not TOMTOM_API_KEY:
        traffic_cache[loc_key] = {}
        return {}

    url = "https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"
    params = {
        "point": f"{loc_key[0]},{loc_key[1]}",
        "unit": "KMPH",
        "key": TOMTOM_API_KEY,
        "openLr": "true",
    }

    try:
        data = _safe_get_json(traffic_session, url, params)
        flow_data = data.get("flowSegmentData") or {}
        row = _flow_segment_to_columns(flow_data)
    except Exception:
        row = {}

    traffic_cache[loc_key] = row
    time.sleep(0.05)
    return row

In [ ]:
temp_list = []
hum_list = []
wind_list = []
traffic_rows = []

for row in tqdm(df.itertuples(index=False), total=len(df), desc=f"Populating chunk {CHUNK_TO_RUN}/{N_CHUNKS}"):
    t, h, w = get_weather_open_meteo(row.lat, row.lon, row.datetime)
    tr = get_traffic_tomtom(row.lat, row.lon)
    temp_list.append(t)
    hum_list.append(h)
    wind_list.append(w)
    traffic_rows.append(tr)

df_out = df.copy()
df_out["Temperature"] = temp_list
df_out["Humidity"] = hum_list
df_out["Wind_speed"] = wind_list

traffic_df = pd.DataFrame(traffic_rows)
df_out = pd.concat([df_out, traffic_df], axis=1)
if "TT_freeFlowSpeed" in df_out.columns:
    df_out["Traffic_flow"] = df_out["TT_freeFlowSpeed"]
else:
    df_out["Traffic_flow"] = np.nan

df_out.to_csv(CHUNK_OUTPUT_CSV, index=False)

print(f"Saved chunk file: {CHUNK_OUTPUT_CSV}")
tomtom_cols = [c for c in df_out.columns if c == "Traffic_flow" or c.startswith("TT_")]
preview_cols = ["datetime", "lat", "lon", "bc", "Temperature", "Humidity", "Wind_speed"] + tomtom_cols
preview_cols = [c for c in preview_cols if c in df_out.columns]
print(df_out[preview_cols].head())
print("\nNull counts in weather + TomTom columns for this chunk:")
print(df_out[["Temperature", "Humidity", "Wind_speed"] + tomtom_cols].isna().sum())

In [ ]:
# Combine all chunk CSVs after all 5 chunks are generated
chunk_files = [
    os.path.join(OUTPUT_DIR, f"extracted_features_chunk_{i:02d}_of_{N_CHUNKS:02d}.csv")
    for i in range(1, N_CHUNKS + 1)
]

missing_chunk_files = [p for p in chunk_files if not os.path.exists(p)]
if missing_chunk_files:
    print("Missing chunk files:")
    for p in missing_chunk_files:
        print(" -", p)
    raise FileNotFoundError("Run all chunks first, then rerun this combine cell.")

combined_df = pd.concat([pd.read_csv(p) for p in chunk_files], ignore_index=True)
combined_df.to_csv(FINAL_OUTPUT_CSV, index=False)

print(f"Combined file saved: {FINAL_OUTPUT_CSV}")
print(f"Rows combined: {len(combined_df)}")
combined_df.head()